# BƯỚC 3: THỬ NGHIỆM CÁC THUẬT TOÁN PHÂN CỤM (CLUSTERING EXPERIMENTS)

## Mục tiêu theo đề bài Practice Exercise 3:
1. **K-Means (Baseline chính):** Quét $K \in [2, 10]$ với `k-means++`, vẽ Elbow & Silhouette để chọn $K$ tối ưu theo dữ liệu thật, thử nghiệm siêu tham số (`init`, `n_init`, `max_iter`).
2. **Hierarchical Clustering (Agglomerative):** So sánh độ bảo toàn khoảng cách Cophenetic (`ward`, `complete`, `average`) và vẽ Dendrogram.
3. **DBSCAN:** Vẽ đồ thị khoảng cách $k$-NN ($k=4$) và quét lưới `(eps, min_samples)` để phát hiện cụm mật độ & điểm nhiễu.
4. **K-Medoids (BUILD tất định & cập nhật nội cụm):** Phân cụm dùng điểm dữ liệu thật làm tâm cụm, bền vững trước ngoại lai.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    K_SEARCH_RANGE,
    NOTEBOOK_FIGURES_DIR,
    NOTEBOOK_METRICS_DIR,
    NOTEBOOK_PROCESSED_DIR,
    NOTEBOOK_SAVED_MODELS_DIR,
    RANDOM_SEED,
    REFERENCE_YEAR,
)
from src.data.preprocessing import compute_train_medians
from src.evaluation.metrics import compare_all_models, evaluate_clustering
from src.evaluation.profiling import create_cluster_profiles, generate_customer_personas
from src.evaluation.visualization import (
    plot_clusters_2d,
    plot_dbscan_k_distance,
    plot_dendrogram,
    plot_elbow_and_silhouette,
)
from src.features.scaling import FeatureScaler
from src.models.dbscan import DBSCANTrainer
from src.models.hierarchical import HierarchicalTrainer
from src.models.kmeans import KMeansTrainer
from src.models.kmedoids import KMedoidsTrainer
from src.models.persistence import save_model

NOTEBOOK_METRICS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(NOTEBOOK_PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(NOTEBOOK_PROCESSED_DIR / "test.csv")
scaler = FeatureScaler(method="standard")
train_matrix = scaler.fit_transform(train_df)
test_matrix = scaler.transform(test_df)

In [ ]:
# 1. K-Means Baseline: Quét K=2..10, vẽ đồ thị và thử nghiệm siêu tham số
kmeans_trainer = KMeansTrainer(random_state=RANDOM_SEED)
k_search_df = kmeans_trainer.search_optimal_k(train_matrix, K_SEARCH_RANGE)
k_search_df.to_csv(NOTEBOOK_METRICS_DIR / "kmeans_k_search.csv", index=False)
optimal_k = int(k_search_df.loc[k_search_df["Silhouette"].idxmax(), "K"])
plot_elbow_and_silhouette(k_search_df, optimal_k=optimal_k, output_dir=NOTEBOOK_FIGURES_DIR)

kmeans_model = kmeans_trainer.fit_final_model(train_matrix, optimal_k=optimal_k)
hyperparam_df = kmeans_trainer.experiment_hyperparameters(train_matrix, n_clusters=optimal_k)
hyperparam_df.to_csv(NOTEBOOK_METRICS_DIR / "kmeans_hyperparams.csv", index=False)
hyperparam_df

In [ ]:
# 2. Hierarchical Clustering (Ward/Complete/Average), quét K=2..10 & Dendrogram
cophenetic_rows = []
hier_trainer = None
for method_name in ("ward", "complete", "average"):
    temp_trainer = HierarchicalTrainer(linkage_method=method_name)
    temp_trainer.compute_linkage_matrix(train_matrix)
    k2_model = temp_trainer.fit_model(train_matrix, n_clusters=2)
    k2_labels = k2_model.labels_
    k2_metrics = evaluate_clustering(
        train_matrix,
        k2_labels,
        model_name=f"Hier_{method_name}_K2",
        compute_dunn=False,
    )
    k2_cluster_counts = pd.Series(k2_labels).value_counts().sort_index().to_dict()
    cophenetic_rows.append(
        {
            "Linkage": method_name,
            "Cophenetic_Correlation": round(temp_trainer.cophenetic_corr_, 4),
            "K2_Silhouette": k2_metrics["silhouette"],
            "K2_Balance_Ratio": k2_metrics["cluster_balance_ratio"],
            "K2_Cluster_Sizes": str(k2_cluster_counts),
        }
    )
    if method_name == "ward":
        hier_trainer = temp_trainer

cophenetic_df = pd.DataFrame(cophenetic_rows)
cophenetic_df.to_csv(NOTEBOOK_METRICS_DIR / "hierarchical_cophenetic.csv", index=False)

hier_search_df = hier_trainer.search_optimal_k(train_matrix, K_SEARCH_RANGE)
hier_search_df.to_csv(NOTEBOOK_METRICS_DIR / "hierarchical_k_search.csv", index=False)
hier_optimal_k = int(hier_search_df.loc[hier_search_df["Silhouette"].idxmax(), "K"])
cut_threshold = hier_trainer.compute_cut_threshold(hier_optimal_k)
plot_dendrogram(hier_trainer.linkage_matrix_, cut_threshold=cut_threshold, output_dir=NOTEBOOK_FIGURES_DIR)
hier_model = hier_trainer.fit_model(train_matrix, n_clusters=hier_optimal_k)
cophenetic_df

In [ ]:
# 3. DBSCAN: Đồ thị k-NN, lưới tham số và biểu đồ cụm mật độ
dbscan_trainer = DBSCANTrainer()
k_dist = dbscan_trainer.compute_k_distances(train_matrix, k=4)
plot_dbscan_k_distance(k_dist, k=4, output_dir=NOTEBOOK_FIGURES_DIR)

dbscan_grid = dbscan_trainer.grid_search_params(
    train_matrix,
    eps_values=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    min_samples_values=[4, 5, 8, 10],
)
dbscan_grid.to_csv(NOTEBOOK_METRICS_DIR / "dbscan_grid_search.csv", index=False)
best_eps, best_ms = dbscan_trainer.best_params_ or (0.5, 5)
dbscan_model = dbscan_trainer.fit_model(train_matrix, eps=best_eps, min_samples=best_ms)
plot_clusters_2d(
    train_df,
    dbscan_model.labels_,
    centroids=None,
    title="Phân Cụm Mật Độ DBSCAN & Các Điểm Nhiễu (-1)",
    filename="02b_dbscan_clusters_2d.png",
    output_dir=NOTEBOOK_FIGURES_DIR,
)
dbscan_grid.head(10)

In [ ]:
# 4. K-Medoids: Dò K=2..10, so sánh thước đo Euclidean/Manhattan và huấn luyện tại K tối ưu
kmedoids_search_df = KMedoidsTrainer(
    n_clusters=optimal_k, metric="euclidean", random_state=RANDOM_SEED
).search_optimal_k(train_matrix, K_SEARCH_RANGE)
kmedoids_search_df.to_csv(NOTEBOOK_METRICS_DIR / "kmedoids_k_search.csv", index=False)
kmedoids_optimal_k = int(kmedoids_search_df.loc[kmedoids_search_df["Silhouette"].idxmax(), "K"])

kmedoids_metric_rows = []
kmedoids_trainer = None
for metric_name in ("euclidean", "manhattan"):
    trainer_km = KMedoidsTrainer(n_clusters=kmedoids_optimal_k, metric=metric_name, random_state=RANDOM_SEED)
    km_labels = trainer_km.fit_predict(train_matrix)
    km_metrics = evaluate_clustering(
        train_matrix,
        km_labels,
        model_name=f"K-Medoids ({metric_name}, K={kmedoids_optimal_k})",
        compute_dunn=False,
    )
    kmedoids_metric_rows.append(
        {
            "Metric": metric_name,
            "K": kmedoids_optimal_k,
            "Total_Cost": round(float(trainer_km.total_cost_), 4),
            "Silhouette": km_metrics["silhouette"],
            "Davies_Bouldin": km_metrics["davies_bouldin"],
            "Calinski_Harabasz": km_metrics["calinski_harabasz"],
        }
    )
    if metric_name == "euclidean":
        kmedoids_trainer = trainer_km

pd.DataFrame(kmedoids_metric_rows).to_csv(NOTEBOOK_METRICS_DIR / "kmedoids_metric_comparison.csv", index=False)
kmedoids_labels = kmedoids_trainer.labels_
kmedoids_search_df

In [ ]:
# 5. Tổng hợp bảng đối chiếu 4 mô hình, kiểm chứng Holdout Test và lưu gói K-Means Baseline phục vụ Notebook 04
notebook_eval_results = [
    evaluate_clustering(train_matrix, kmeans_model.labels_, f"K-Means (K={optimal_k})"),
    evaluate_clustering(train_matrix, hier_model.labels_, f"Hierarchical Ward (K={hier_optimal_k})"),
    evaluate_clustering(train_matrix, dbscan_model.labels_, f"DBSCAN (eps={best_eps}, ms={best_ms})"),
    evaluate_clustering(train_matrix, kmedoids_labels, f"K-Medoids (K={kmedoids_optimal_k})"),
]
comparison_df = compare_all_models(
    notebook_eval_results,
    output_path=NOTEBOOK_METRICS_DIR / "model_comparison.csv",
)

kmeans_test_labels = kmeans_model.predict(test_matrix)
kmedoids_test_labels = kmedoids_trainer.predict(test_matrix)
holdout_df = pd.DataFrame(
    [
        evaluate_clustering(test_matrix, kmeans_test_labels, f"K-Means Test Holdout (K={optimal_k})"),
        evaluate_clustering(test_matrix, kmedoids_test_labels, f"K-Medoids Test Holdout (K={kmedoids_optimal_k})"),
    ]
)
holdout_df.to_csv(NOTEBOOK_METRICS_DIR / "holdout_test_evaluation.csv", index=False)

profile_df = create_cluster_profiles(
    train_df,
    kmeans_model.labels_,
    output_path=NOTEBOOK_METRICS_DIR / "cluster_profiles.csv",
)
personas = generate_customer_personas(profile_df)
save_model(
    kmeans_model,
    scaler,
    model_name="final_kmeans_pipeline.pkl",
    metadata={
        "feature_cols": scaler.feature_cols,
        "scaler_method": scaler.method,
        "n_clusters": int(optimal_k),
        "reference_year": REFERENCE_YEAR,
        "feature_medians": compute_train_medians(train_df),
        "personas": personas,
    },
    output_dir=NOTEBOOK_SAVED_MODELS_DIR,
)
comparison_df